# CANN、芯片以及 PyPTO 介绍

本章是 PyPTO 算子开发学习路径的起点。正式编写算子之前，需要先建立三层基础认知：昇腾芯片提供什么样的硬件能力，CANN 在软件栈中承担什么职责，PyPTO 又如何把开发者的算法表达落到芯片执行。

可以先用一句话把三者串起来：**昇腾芯片提供 AI 算力，CANN 提供连接硬件与上层框架的软件栈，PyPTO 提供面向算子开发的 Python 编程框架**。后续章节会围绕这条主线展开。

本节不涉及代码实践，重点是帮助读者建立全局地图。理解这张地图后，再阅读 CANN、芯片架构和 PyPTO 编程模型时，就能更容易判断每个概念处在什么层级、解决什么问题。

## 1. 三者在开发链路中的位置

学习 PyPTO 时，经常会同时遇到 CANN、昇腾芯片和 PyPTO 这三个概念。它们不是并列关系，而是从硬件到软件再到开发接口的分层关系：

- **昇腾芯片（NPU）**：硬件层，提供 AI 计算所需的物理算力，包括 AI Core、多级片上存储、Global Memory 以及多精度计算能力。
- **CANN（Compute Architecture for Neural Networks）**：软件栈层，提供驱动固件、Toolkit、Ops 包、编译优化工具链、运行时调度和性能分析等能力，是上层框架访问昇腾硬件的基础。
- **PyPTO**：编程框架层，基于 CANN 的能力，为开发者提供 Python 友好的算子开发接口，通过 PTO 编程范式和 Tile 编程模型描述计算逻辑。

它们的关系可以概括为：

```text
开发者
  |
  v
PyPTO：用 Python 和 Tensor/Tile 抽象描述“算什么”
  |
  v
CANN：完成编译优化、代码生成、运行时调度和工具链支持
  |
  v
昇腾芯片：在 AI Core 和内存层次上真正执行计算
```

也就是说，开发者通常通过 PyPTO 表达计算，CANN 负责把这种表达逐步 lowering、优化并调度执行，最终由昇腾芯片完成实际计算。

## 2. 为什么需要先理解芯片和 CANN

PyPTO 的核心价值，是让开发者不必直接编写底层硬件指令，也能完成高性能算子开发。但 PyPTO 并不是脱离硬件的纯数学描述，它的很多设计都来自昇腾硬件和 CANN 编译执行体系。

例如：

- PyPTO 强调 **Tile 编程模型**，是因为芯片侧存在多级内存层次和并行计算单元，需要把大 Tensor 切分成适合搬运、缓存和计算的数据块。
- PyPTO 支持多层级计算图，例如 Tensor Graph、Tile Graph、Block Graph 和 Execute Graph，是为了让高层算法表达能够逐步转换为硬件可执行的调度与指令序列。
- CANN 提供 Toolkit、Ops 包、编译链和运行时环境，因此 PyPTO 编写出的算子并不是孤立运行，而是进入 CANN 软件栈完成编译、优化、加载与执行。
- 昇腾芯片支持多精度计算、MPMD 执行模式和 AI Core 并行执行，这些能力决定了算子优化时需要关注数据类型、并行粒度、内存访问和任务调度。

因此，本章先从概念层面把硬件、软件栈和编程框架的边界讲清楚，为后续学习算子表达、编译流程和性能优化打基础。

## 3. 本章学习目标

完成本章后，读者应能够：

1. 说明 CANN 在昇腾 AI 开发体系中的定位，以及驱动固件、Toolkit、Ops 包等核心组成的作用。
2. 了解昇腾芯片面向 AI 计算的基本特征，包括 AI Core、多级内存、精度支持和 MPMD 执行模式。
3. 理解 PyPTO 的定位：它不是通用深度学习框架，而是面向昇腾高性能算子开发的编程框架。
4. 理解 PTO 编程范式和 Tile 编程模型的基本思想，知道为什么 PyPTO 要围绕 Tensor 和 Tile 组织计算。
5. 说清楚“芯片 - CANN - PyPTO - 开发者代码”之间的依赖关系。
6. 初步判断 PyPTO 适合哪些场景，例如自定义算子、融合算子、大模型组件和动态 Shape 处理。

## 4. 章节路线

本章共四节，建议按顺序阅读：

| 小节 | 主题 | 关注重点 |
| --- | --- | --- |
| 1.1 | 章节介绍 | 建立 CANN、昇腾芯片和 PyPTO 的整体关系 |
| 1.2 | CANN | 软件栈定位、核心组成、功能特性和安装方式 |
| 1.3 | 昇腾芯片 | AI Core、内存层次、精度支持、MPMD 执行模式和应用场景 |
| 1.4 | PyPTO | PTO 编程范式、Tile 编程模型、多层级计算图、核心特性和适用场景 |

阅读顺序很重要。先理解 CANN，才能知道 PyPTO 生成的内容如何进入编译和运行流程；先理解芯片的计算与内存结构，才能理解 Tile 为什么是 PyPTO 的关键抽象；最后再看 PyPTO，才能把编程模型与底层执行机制对应起来。

## 5. 阅读时可以带着的问题

本章偏概念导入，阅读时可以重点思考以下问题：

1. CANN 在芯片和上层开发框架之间承担了哪些职责？
2. 昇腾芯片的内存层次和 AI Core 并行能力，为什么会影响算子的编程模型？
3. PyPTO 的 Tile 编程模型解决了什么问题？它和硬件内存搬运、并行计算有什么关系？
4. Tensor Graph、Tile Graph、Block Graph、Execute Graph 分别更接近算法表达、硬件感知优化还是最终执行？
5. 作为算子开发者，为什么通常希望使用 PyPTO，而不是直接面对底层硬件指令？

这些问题会在后续小节逐步得到回答。读者不需要在本节掌握所有细节，只需要先建立“从算法表达到硬件执行”的整体视角。

## 6. 本节小结

本节完成了本章导读：昇腾芯片是硬件算力基础，CANN 是连接硬件与上层框架的软件栈，PyPTO 是面向高性能算子开发的 Python 编程框架。

接下来将分别介绍 CANN、昇腾芯片和 PyPTO。学完这三部分后，读者应能从全栈角度理解 PyPTO 算子开发：代码如何被描述，如何被 CANN 编译优化，又如何最终运行在昇腾芯片上。